In [2]:
import os
from dotenv import load_dotenv
import requests

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent

load_dotenv()

@tool
def get_current_time(city: str) -> str:
    """Get the current local time for a city in the world."""

    url = "https://timezones.live/api/v1/time"

    response = requests.get(
        url,
        params={"city": city}
    )

    response.raise_for_status()

    data = response.json()["data"]

    return (
        f"Current time in {data['city']}, "
        f"{data['country']} is {data['currentTime']} "
        f"on {data['currentDate']}."
    )


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together."""

    return a + b


llm = ChatGroq(
    model=os.environ.get("GROQ_MODEL_OPEN"),
    temperature=0
)


tools = [
    get_current_time,
    add_numbers
]


agent = create_agent(
    model=llm,
    tools=tools
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What time is it in Tokyo?"
        }
    ]
})


print(result["messages"][-1].content)

for message in result["messages"]:
    print("=" * 70)
    print("MESSAGE TYPE:", type(message).__name__)
    print(message)

It’s currently **03:05 AM on Sunday, September 13** in Tokyo.
MESSAGE TYPE: HumanMessage
content='What time is it in Tokyo?' additional_kwargs={} response_metadata={} id='460a0f21-0691-4619-8318-020062e72338'
MESSAGE TYPE: AIMessage
content='' additional_kwargs={'reasoning_content': 'We need to call get_current_time with city "Tokyo".', 'tool_calls': [{'id': 'fc_d3ae5c21-ea5b-46a9-bdfb-3635210bd2cc', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_current_time'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 157, 'total_tokens': 194, 'completion_time': 0.037553864, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.007748096, 'prompt_tokens_details': None, 'queue_time': 0.308824634, 'total_time': 0.04530196}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3cdd24b83b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a09

In [3]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 125 + 375?"
        }
    ]
})

print(result["messages"][-1].content)

for message in result["messages"]:
    print("=" * 70)
    print(type(message).__name__)
    print(message)

125 + 375 = **500**
HumanMessage
content='What is 125 + 375?' additional_kwargs={} response_metadata={} id='bc816cbb-c78e-414a-ae5b-d8e993980fe0'
AIMessage
content='' additional_kwargs={'reasoning_content': "We need to add 125 + 375. That's 500. We can use add_numbers function.", 'tool_calls': [{'id': 'fc_fa5bc6af-7642-43d3-ad60-b768544f22f1', 'function': {'arguments': '{"a":125,"b":375}', 'name': 'add_numbers'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 158, 'total_tokens': 207, 'completion_time': 0.051051068, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.007845896, 'prompt_tokens_details': None, 'queue_time': 0.317351713, 'total_time': 0.058896964}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a096cc-b43a-7c93-bd75-df8c0668de3d-0' tool_calls=[{'name': 'add_